# NES-VMC for He Atom (K=2) - 6-31G Basis

使用 NES-VMC 算法计算 He 原子的前 K=2 个激发态能量
基组: 6-31G (分裂价基组)

In [3]:
import jax
import jax.numpy as jnp
import flax.nnx as nnx
import netket as nk
import netket.experimental as nkx
import sys
sys.path.append('..')
from NES_VMC import NESTotalAnsatz, create_machine,\
    ha, SingleStateAnsatz, create_single_machine,\
        create_machine_matrix, Ham_psi, Ham_Psi, NES_loss_energy, nes_vmc_gradient, hi, E_fcis,\
        NESFermionHopRule, compute_qgt, sampler_info
import optax
from functools import partial
from jax.flatten_util import ravel_pytree
import time
from pyscf import gto, scf, fci

## 1. He 原子设置 (6-31G 基组)

In [4]:
# ===================== He 原子定义 & FCI 基准 (6-31G) =====================
geometry = [('He', (0., 0., 0.))]
mol = gto.M(atom=geometry, basis='6-31G', verbose=0)
mf = scf.RHF(mol).run(verbose=0)

# FCI 精确基准
cisolver = fci.FCI(mf)
cisolver.nroots = 4
E_fcis, fcivec = cisolver.kernel()
print("="*60)
print("He 原子 FCI 基准能量 (6-31G 基组)")
print("="*60)
for i, e in enumerate(E_fcis):
    exc = (e - E_fcis[0]) * 27.2114
    print(f"E{i} = {e:.8f} Ha  |  激发能：{exc:.4f} eV")

print(f"\n基函数数目: {mol.nao_nr()}")
print(f"电子数: {mol.nelectron}")

He 原子 FCI 基准能量 (6-31G 基组)
E0 = -2.87016214 Ha  |  激发能：0.0000 eV
E1 = -1.39930780 Ha  |  激发能：40.0240 eV
E2 = -0.94871288 Ha  |  激发能：52.2853 eV
E3 = 0.60863701 Ha  |  激发能：94.6630 eV

基函数数目: 2
电子数: 2


In [5]:
# ===================== He 原子的 NetKet 哈密顿量和希尔伯特空间 =====================
ha = nkx.operator.from_pyscf_molecule(mol)

# He 原子 6-31G 基组:
# - 1s 内层: 1 个基函数
# - 1s 分裂价: 2 个基函数 (但实际只有2个空间轨道)
# 总共 2 个空间轨道，4 个自旋轨道
N_ORBITALS = mol.nao_nr()  # 6-31G 基组下 He 有 2 个空间轨道
N_FERMIONS_PER_SPIN = (1, 1)  # 1 alpha + 1 beta 电子

hi = nk.hilbert.SpinOrbitalFermions(
    n_orbitals=N_ORBITALS,
    s=1/2,
    n_fermions_per_spin=N_FERMIONS_PER_SPIN,
)

K = 2  # NES 扩展副本数 (计算前 2 个激发态)
hi_ext = hi ** K  # 扩展希尔伯特空间
SINGLE_SIZE = hi.size  # 单个子系统维度 = 4 (2 轨道 × 2 自旋)

print(f"空间轨道数: {N_ORBITALS}")
print(f"自旋轨道数 (n_spin_orbitals): {SINGLE_SIZE}")
print(f"单粒子希尔伯特空间维度: {hi.size}")
print(f"扩展希尔伯特空间维度: {hi_ext.size}")

空间轨道数: 2
自旋轨道数 (n_spin_orbitals): 4
单粒子希尔伯特空间维度: 4
扩展希尔伯特空间维度: 8


## 2. He 原子的费米子跃迁边

对于 6-31G 基组的 He 原子，有 2 个空间轨道（1s 内层 + 分裂价 1s'）。
edges 定义了允许的费米子跃迁（需要满足粒子数守恒和自旋守恒）。

In [6]:
# 6-31G He: 2 个空间轨道，每个轨道有 alpha 和 beta 两个电子态
# 自旋顺序: [alpha_0, beta_0, alpha_1, beta_1]
# 轨道索引:  0          1          2          3
#
# 费米子跃迁边 (同一轨道内 alpha<->beta 交换):
# - (0, 1): 轨道 0 的 alpha <-> beta
# - (2, 3): 轨道 1 的 alpha <-> beta
single_edges = ((0, 1), (2, 3))

g = nk.graph.Graph(edges=single_edges)
single_rule = nk.sampler.rules.FermionHopRule(hi, graph=g)
tensor_rule = nk.sampler.rules.TensorRule(hi_ext, [single_rule] * K)

print(f"单系统跃迁边: {single_edges}")
print(f"单系统希尔伯特空间大小 (n_spin_orbitals): {SINGLE_SIZE}")
print(f"扩展希尔伯特空间大小: {K * SINGLE_SIZE}")

单系统跃迁边: ((0, 1), (2, 3))
单系统希尔伯特空间大小 (n_spin_orbitals): 4
扩展希尔伯特空间大小: 8


## 3. 初始化 Ansatz

In [7]:
# He 原子 6-31G 的 n_spin_orbitals = 4 (2 轨道 × 2 自旋)
N_SPIN_ORBITALS = hi.size  # = 4
HIDDEN_DIM = 16  # 隐藏层维度

total_ansatz = NESTotalAnsatz(N_SPIN_ORBITALS, K, HIDDEN_DIM, rngs=nnx.Rngs(11))
total_machine, total_graphdef, total_params = create_machine(total_ansatz)
total_matrix_machine, total_graphdef, total_params = create_machine_matrix(total_ansatz)

single_machine_list = []
for ansatz in total_ansatz.single_ansatz_list:
    m, g, p = create_single_machine(ansatz)
    single_machine_list.append(m)

print(f"Ansatz 参数总数: {len(ravel_pytree(total_params)[0])}")

Ansatz 参数总数: 738


## 4. 采样器设置

In [8]:
import jax
import jax.numpy as jnp
import netket as nk

@nk.utils.struct.dataclass
class NESFermionHopRule(nk.sampler.rules.MetropolisRule):
    edges: jnp.ndarray
    K: int = nk.utils.struct.static_field()
    single_size: int = nk.utils.struct.static_field()

    def _check_duplicate(self, sigma_ext):
        """NES约束：子组态不重复"""
        sub = sigma_ext.reshape((-1, self.K, self.single_size))
        return jnp.any(jnp.all(sub[...,1:,:] == sub[...,0:1,:], axis=-1), axis=-1).squeeze()

    def transition(self, sampler, machine, parameters, state, rng, sigma):
        """跃迁规则"""
        batch_size = sigma.shape[0]
        key1, key2 = jax.random.split(rng)

        e_idx = jax.random.randint(key1, (batch_size,), 0, self.edges.shape[0])
        sel_e = self.edges[e_idx]
        i, j = sel_e[:,0], sel_e[:,1]

        sigma_cand = sigma.at[jnp.arange(batch_size),i].set(sigma[jnp.arange(batch_size),j])
        sigma_cand = sigma_cand.at[jnp.arange(batch_size),j].set(sigma[jnp.arange(batch_size),i])

        invalid = self._check_duplicate(sigma_cand)
        new_sigma = jnp.where(invalid[:, None], sigma, sigma_cand)

        return new_sigma, None

    def random_state(self, sampler, machine, parameters, state, rng):
        """随机态生成"""
        sigma_shape = state.σ.shape
        hilbert = sampler.hilbert

        def gen_single(key):
            max_tries = 100
            def cond(c): 
                return (c[0] < max_tries) & c[2]
            
            def body(c):
                tries, k, _, _ = c
                k, k_new = jax.random.split(k)
                s = hilbert.random_state(k_new)
                is_dup = self._check_duplicate(s)
                return (tries + 1, k, is_dup, s)
            
            init_c = (0, key, True, hilbert.random_state(key))
            final_c = jax.lax.while_loop(cond, body, init_c)
            tries, _, is_dup, s = final_c
            return jax.lax.cond(is_dup, lambda: hilbert.random_state(key), lambda: s)
        
        keys = jax.random.split(rng, sigma_shape[0])
        return jax.vmap(gen_single)(keys)

In [9]:
N_CHAINS = 16
N_WARMUP = 100
N_SAMPLES_PER_CHAIN = 200
SWEEP_SIZE = 30

# 构建扩展希尔伯特空间的跃迁边
ext_edges = []
for k in range(K):
    offset = k * SINGLE_SIZE
    for (i, j) in single_edges:
        ext_edges.append((i + offset, j + offset))
ext_edges = jnp.array(ext_edges)
print(f"扩展希尔伯特空间跃迁边数: {len(ext_edges)}")

nes_rule = NESFermionHopRule(edges=ext_edges, K=K, single_size=SINGLE_SIZE)
nes_sampler = nk.sampler.MetropolisSampler(
    hilbert=hi_ext,
    rule=nes_rule,
    n_chains=N_CHAINS,
    sweep_size=20,
)

# 采样器状态初始化
sampler_rng = jax.random.PRNGKey(21)
sampler_state = nes_sampler.init_state(total_machine, total_params, sampler_rng)

# 测试采样
samples_raw, sampler_state = nes_sampler.sample(
    total_machine, total_params, state=sampler_state, chain_length=40
)
print(f"采样形状: {samples_raw.shape}")  # (n_chains, chain_length, K * SINGLE_SIZE)

扩展希尔伯特空间跃迁边数: 4
采样形状: (16, 40, 8)


## 5. 训练循环

In [10]:
N_ITER = 400
Natural_Grad = False  # 设置为 True 则使用自然梯度

optimizer = optax.sgd(learning_rate=0.01)
opt_state = optimizer.init(total_params)

print("\n" + "="*60)
print("开始多链 NES-VMC 训练 (NetKet 自定义采样器)")
print("="*60)
print(f"基态能量={E_fcis[0]:.8f} Ha| 第一激发态能量={E_fcis[1]:.8f} Ha| 第二激发态能量={E_fcis[2]:.8f} Ha")


开始多链 NES-VMC 训练 (NetKet 自定义采样器)
基态能量=-2.87016214 Ha| 第一激发态能量=-1.39930780 Ha| 第二激发态能量=-0.94871288 Ha


In [ ]:
history = {
    'step': [],
    'energy_0st': [],
    'energy_1st': [],
    'energy_std': [],
    'loss': [],
    'params': [],
    'E_Lmatrix':[],
    'grad_flat':[],
    'samples':[],
    'log_Psi_mean':[],
    'log_Psi_min':[],
    'log_Psi_max':[],
    'grad_norm':[],
}

start_time = time.time()
for step in range(N_ITER):
    # 1. 采样
    samples_raw, sampler_state = nes_sampler.sample(
        machine=total_machine, parameters=total_params, 
        state=sampler_state, chain_length=N_SAMPLES_PER_CHAIN
    )
    samples = samples_raw.reshape(-1, hi_ext.size)
    x_batch = samples.reshape(-1, K, N_SPIN_ORBITALS)
    
    # 2. 计算梯度
    grad, loss_mean, E_L_mean = nes_vmc_gradient(
        ha=ha,
        total_matrix_machine=total_matrix_machine,
        total_machine=total_machine,
        single_machine_list=single_machine_list,
        total_params=total_params,
        x_batch=x_batch
    )
    
    grad_flat, grad_unravel_fn = ravel_pytree(grad)
    
    # 3. 自然梯度 (可选)
    if Natural_Grad == True:
        qgt_reg, unravel_fn = compute_qgt(
            total_machine, total_params, x_batch, diag_shift=0.1
        )
        natural_grad_flat = jnp.linalg.solve(qgt_reg, grad_flat)
        grad = grad_unravel_fn(natural_grad_flat)
    
    # 4. 更新参数
    updates, opt_state = optimizer.update(grad, opt_state, total_params)
    total_params = optax.apply_updates(total_params, updates)
    
    # 5. 记录历史
    log_Psi_batch = total_machine(total_params, x_batch)
    eig_vals, eig_vecs = jnp.linalg.eigh(E_L_mean)
    grad_norm = jnp.linalg.norm(grad_flat)
    
    history['step'].append(step)
    history['E_Lmatrix'].append(E_L_mean)
    history['samples'].append(samples)
    history['loss'].append(loss_mean)
    history['log_Psi_mean'].append(log_Psi_batch.mean())
    history['log_Psi_min'].append(log_Psi_batch.min())
    history['log_Psi_max'].append(log_Psi_batch.max())
    history['grad_norm'].append(grad_norm)
    history['energy_0st'].append(eig_vals[0])
    history['energy_1st'].append(eig_vals[1])
    history['params'].append(total_params)
    
    # 6. 定期输出
    if step % 50 == 0 or step == N_ITER - 1:
        print(f"log_Psi: mean={log_Psi_batch.mean():.3f} | min={log_Psi_batch.min():.3f} | max={log_Psi_batch.max():.3f}")
        print(f"grad norm = {grad_norm:.4f}")
        print(f"Step {step:3d} | Loss: {loss_mean}|0st能量={eig_vals[0]:.8f} Ha｜1st能量={eig_vals[1]:.8f} Ha")
        print('#-----------------------------------------#')

end_time = time.time()
print(f"训练耗时：{end_time - start_time:.2f} 秒")
print("\n" + "="*60)
print("训练完成!")
print("="*60)

## 6. 结果可视化

In [ ]:
import matplotlib.pyplot as plt

fig, axs = plt.subplots(2, 2, figsize=(12, 9))
fig.suptitle('NES-VMC for He Atom K=2 (6-31G Basis)')

# 基态能量
axs[0, 0].plot(history['energy_0st'], color='orange', label='NES-VMC')
axs[0, 0].hlines(E_fcis[0], 0, len(history['energy_0st']), linestyle='--', color='red', label='FCI')
axs[0, 0].set_title('Ground State Energy (E0)')
axs[0, 0].set_ylabel('energy (Ha)')
axs[0, 0].set_xlabel('step')
axs[0, 0].legend()

# 第一激发态
axs[0, 1].plot(history['energy_1st'], color='orange', label='NES-VMC')
axs[0, 1].hlines(E_fcis[1], 0, len(history['energy_1st']), linestyle='--', color='red', label='FCI')
axs[0, 1].set_title('First Excited State Energy (E1)')
axs[0, 1].set_ylabel('energy (Ha)')
axs[0, 1].set_xlabel('step')
axs[0, 1].legend()

# Loss
axs[1, 0].plot(history['loss'], color='blue', label='Loss')
axs[1, 0].set_title('Loss Function')
axs[1, 0].set_ylabel('loss')
axs[1, 0].set_xlabel('step')
axs[1, 0].legend()

# Grad Norm
axs[1, 1].plot(history['grad_norm'], color='green', label='Grad Norm')
axs[1, 1].set_title('Gradient Norm')
axs[1, 1].set_ylabel('grad norm')
axs[1, 1].set_xlabel('step')
axs[1, 1].legend()

plt.tight_layout()
plt.show()

## 7. 误差分析

In [ ]:
print("\n" + "="*60)
print("最终误差分析")
print("="*60)

final_E0 = history['energy_0st'][-1]
final_E1 = history['energy_1st'][-1]

print(f"基态能量 (NES-VMC): {final_E0:.8f} Ha")
print(f"基态能量 (FCI):     {E_fcis[0]:.8f} Ha")
print(f"基态误差:           {abs(final_E0 - E_fcis[0]):.8f} Ha ({abs(final_E0 - E_fcis[0])*27.2114:.4f} eV)")
print()
print(f"第一激发态 (NES-VMC): {final_E1:.8f} Ha")
print(f"第一激发态 (FCI):     {E_fcis[1]:.8f} Ha")
print(f"第一激发态误差:       {abs(final_E1 - E_fcis[1]):.8f} Ha ({abs(final_E1 - E_fcis[1])*27.2114:.4f} eV)")
print()
print(f"FCI 激发能: {E_fcis[1] - E_fcis[0]:.8f} Ha ({(E_fcis[1] - E_fcis[0])*27.2114:.4f} eV)")
print(f"NES-VMC 激发能: {final_E1 - final_E0:.8f} Ha ({(final_E1 - final_E0)*27.2114:.4f} eV)")